In [4]:
import pandas as pd
from pathlib import Path
import glob

# =====================================================
# CONFIGURACION
# =====================================================

RUTA_BASE = r"C:\Users\NI38504\Documents\Home\REMESAS\INSUMOS\Julio"
RUTA_BASE = r"\\172.25.16.15\Reportes Cobis_BI\Remesas\2026\Julio"
RUA_DESTINO = R"C:\Users\NI38504\Documents\Home\REMESAS\INSUMOS\Julio\CONSOLIDADO"

salida_excel = Path(RUA_DESTINO) / "Consolidado_Pairpak_di.xlsx"

# DataFrames finales
df_pagadas_total = pd.DataFrame()
df_enviadas_total = pd.DataFrame()

# =====================================================
# RECORRER CARPETAS
# =====================================================

for carpeta in Path(RUTA_BASE).iterdir():

    if not carpeta.is_dir():
        continue

    archivos = glob.glob(str(carpeta / "pairpak_di*.xls"))

    if not archivos:
        print(f"No se encontró archivo en {carpeta.name}")
        continue

    archivo = archivos[0]

    print(f"Procesando: {archivo}")

    # =====================================================
    # LEER ARCHIVO
    # =====================================================

    with open(
        archivo,
        "r",
        encoding="latin-1",
        errors="ignore"
    ) as f:

        lineas = [line.strip() for line in f.readlines()]

    # =====================================================
    # BUSCAR SECCIONES
    # =====================================================

    idx_pagadas = None
    idx_enviadas = None

    for i, linea in enumerate(lineas):

        texto = linea.upper()

        if "REMESAS PAGADAS" in texto:
            idx_pagadas = i

        elif "REMESAS ENVIADAS" in texto:
            idx_enviadas = i

    if idx_pagadas is None or idx_enviadas is None:
        print(
            f"No se encontraron ambas etiquetas en {carpeta.name}"
        )
        continue

    # =====================================================
    # REMESAS PAGADAS
    # =====================================================

    encabezado_pagadas = lineas[idx_pagadas + 1].split("|")

    registros_pagadas = []

    for linea in lineas[idx_pagadas + 2 : idx_enviadas]:

        if not linea.strip():
            continue

        fila = linea.split("|")

        if len(fila) == len(encabezado_pagadas):
            registros_pagadas.append(fila)

    if registros_pagadas:

        df_pagadas = pd.DataFrame(
            registros_pagadas,
            columns=encabezado_pagadas
        )

        df_pagadas["CARPETA_ORIGEN"] = carpeta.name

        df_pagadas_total = pd.concat(
            [df_pagadas_total, df_pagadas],
            ignore_index=True
        )

    # =====================================================
    # REMESAS ENVIADAS
    # =====================================================

    encabezado_enviadas = lineas[idx_enviadas + 1].split("|")

    registros_enviadas = []

    for linea in lineas[idx_enviadas + 2:]:

        if not linea.strip():
            continue

        fila = linea.split("|")

        if len(fila) == len(encabezado_enviadas):
            registros_enviadas.append(fila)

    if registros_enviadas:

        df_enviadas = pd.DataFrame(
            registros_enviadas,
            columns=encabezado_enviadas
        )

        df_enviadas["CARPETA_ORIGEN"] = carpeta.name

        df_enviadas_total = pd.concat(
            [df_enviadas_total, df_enviadas],
            ignore_index=True
        )

# =====================================================
# EXPORTAR
# =====================================================

with pd.ExcelWriter(
    salida_excel,
    engine="openpyxl"
) as writer:

    df_pagadas_total.to_excel(
        writer,
        sheet_name="REMESAS PAGADAS",
        index=False
    )

    df_enviadas_total.to_excel(
        writer,
        sheet_name="REMESAS ENVIADAS",
        index=False
    )

print("Proceso finalizado.")
print(f"Archivo generado: {salida_excel}")

Procesando: \\172.25.16.15\Reportes Cobis_BI\Remesas\2026\Julio\01072026\pairpak_di_0101072026.xls
Procesando: \\172.25.16.15\Reportes Cobis_BI\Remesas\2026\Julio\02072026\pairpak_di_0202072026.xls
Procesando: \\172.25.16.15\Reportes Cobis_BI\Remesas\2026\Julio\03072026\pairpak_di_0303072026.xls
Procesando: \\172.25.16.15\Reportes Cobis_BI\Remesas\2026\Julio\04072026\pairpak_di_0404072026.xls
Procesando: \\172.25.16.15\Reportes Cobis_BI\Remesas\2026\Julio\05072026\pairpak_di_0505072026.xls
Procesando: \\172.25.16.15\Reportes Cobis_BI\Remesas\2026\Julio\06072026\pairpak_di_0606072026.xls
Procesando: \\172.25.16.15\Reportes Cobis_BI\Remesas\2026\Julio\07072026\pairpak_di_0707072026.xls
Procesando: \\172.25.16.15\Reportes Cobis_BI\Remesas\2026\Julio\08072026\pairpak_di_0808072026.xls
Procesando: \\172.25.16.15\Reportes Cobis_BI\Remesas\2026\Julio\09072026\pairpak_di_0909072026.xls
Procesando: \\172.25.16.15\Reportes Cobis_BI\Remesas\2026\Julio\10072026\pairpak_di_1010072026.xls
Procesando